## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">Food Recommendation System</p>

<a id="toc"></a>

## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">Content</p>

* [INTRODUCTION](#0)
* [IMPORTING MODULES, LOADING DATA & DATA REVIEW](#1)
* [PREPROCESSING](#2)
* [EXPLORATORY DATA ANALYSIS (EDA)](#3)    
* [SCALING, CATEGORICAL VARIABLES, SPLITTING](#4)
* [MODELS](#5)
* [CONCLUSION](#6)

## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">Introduction</p>

<a id="0"></a>
<a href="#toc" class="btn btn-primary btn-sm" role="button" aria-pressed="true"
style="color:blue; background-color:#dfa8e4" data-toggle="popover">Content</a>

## 1.1 Information About the Project

**Objective:**  
The primary objective of this project is to develop a recommendation system for a large food company to mitigate customer information overload. Since the dataset contains no explicit ratings, the core challenge is to build, compare, and evaluate multiple recommendation approaches by deriving implicit customer preferences from their transactional behavior.

**Scope:**  
This project goes beyond simply writing code; it focuses on thinking like a data scientist by making strategic decisions, evaluating trade-offs, and justifying choices with evidence. The scope encompasses:

* Exploring and preprocessing real-world, sparse data.

* Engineering an implicit rating strategy based on factors like frequency, spending, or recency.

* Implementing and comparing at least two different recommendation approaches (e.g., User-Based Collaborative Filtering, Item-Based Collaborative Filtering, Popularity Baseline).

* Evaluating methods using ranking metrics like Precision and Recall rather than traditional error metrics.

* Connecting the technical results to business value through deployment recommendations and addressing business constraints.

## 1.2 Description of the Dataset

- **Source:** The dataset (recom.csv) is a real transactional dataset provided by a large food company.

- **Size:** It contains approximately 50,000 transactions from 28,514 unique customers across 152 unique items.

- **Type:** This is tabular transactional data containing implicit feedback, collected over a 3-month period (August 26 to December 3, 2022). It is characterized by extreme sparsity (the median customer has only 1 transaction) and a high variance in item popularity.

## 1.3 Description of the Columns

- **Target Variable:** There is NO explicit target variable (like stars or numerical scores) in this dataset. Instead, an implicit rating is engineered using the behavioral signals provided in the data, such as purchase frequency, recency, spending, or simple binary purchase signals.

- **Feature Variables:** 

    * Main ID: The customer identifier (representing 28,514 unique customers)

    * Transaction ID: The unique identifier for each transaction.

    * Date: The transaction timestamp (August 26 - December 3, 2022).

    * Price: The price of the purchased item, ranging from $0.50 to $2,525.50.

    * Code Product: The product code, which represents 333 unique products.

    * Amount: The quantity of items purchased in the transaction, ranging from 1 to 54 units.

    * ItemKey: The item identifier for 152 unique items. Note: 43% of these values are missing and will require a strategic handling decision.


## <p style="background-color:#9d4f8c; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">Importing Modules, Load Data & Data Review</p>

<a id="1"></a>
<a href="#toc" class="btn btn-primary btn-sm" role="button" aria-pressed="true" 
style="color:blue; background-color:#dfa8e4" data-toggle="popover">Content</a>

In [18]:
import pandas as pd
import numpy as np
from IPython.display import display

In [19]:
# load the data
df = pd.read_csv('recom.csv')

df.head().T

,0,1,2,3,4
Unnamed: 0,0,1,2,3,4
Main_ID,90fada91,9006f9ac,32270891,97e03e47,41949228
Transaction_ID,264f7a69,45c7d853,61ad76dd,41ee09f6,244fe6d8
Date,2022-10-07 20:53:49.153,2022-09-17 15:54:57.187,2022-11-28 13:51:55.667,2022-09-12 16:20:22.110,2022-10-14 18:53:43.933
Price,125.0,19.0,141.0,4.5,129.5
Code_Product,5002.0,35012.0,5005.0,35078.5,49291.5
Amount,1.0,1.0,1.0,1.0,5.0
ItemKey,5002.0,NaN,5005.0,NaN,NaN


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      50000 non-null  int64  
 1   Main_ID         50000 non-null  str    
 2   Transaction_ID  50000 non-null  str    
 3   Date            50000 non-null  str    
 4   Price           50000 non-null  float64
 5   Code_Product    50000 non-null  float64
 6   Amount          50000 non-null  float64
 7   ItemKey         28597 non-null  float64
dtypes: float64(4), int64(1), str(3)
memory usage: 3.1 MB


In [21]:
df.nunique()

Unnamed: 0        50000
Main_ID           28514
Transaction_ID    48403
Date              48398
Price               795
Code_Product        333
Amount               22
ItemKey             152
dtype: int64

In [22]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,50000.0,24999.500000,14433.901067,0.0,12499.75,24999.5,37499.25,49999.0
Price,50000.0,62.560670,68.269624,0.5,24.50,45.5,83.00,2525.5
Code_Product,50000.0,32379.293540,21697.500334,5000.5,10013.00,40009.5,49291.50,350027.5
Amount,50000.0,1.232640,0.749353,1.0,1.00,1.0,1.00,54.0
ItemKey,28597.0,20775.740952,16481.882853,5000.5,5011.50,10023.0,40028.50,57035.5


In [23]:
df.duplicated().sum()

np.int64(0)

### Standardize Column Names

In [24]:
df.columns = (
    df.columns
    # Handle CamelCase (turns 'ItemKey' into 'Item_Key')
    .str.replace(r'([a-z])([A-Z])', r'\1_\2', regex=True) 
    
    # Replace spaces, colons, or any other non-alphanumeric character with an underscore
    .str.replace(r'[^a-zA-Z0-9]', '_', regex=True) 
    
    # Clean up any accidental double underscores (like from 'Unnamed: 0')
    .str.replace(r'_+', '_', regex=True)
    
    # Make everything lowercase
    .str.lower()
    
    # Strip any trailing or leading underscores just to be safe
    .str.strip('_') 
)

df.columns

Index(['unnamed_0', 'main_id', 'transaction_id', 'date', 'price',
       'code_product', 'amount', 'item_key'],
      dtype='str')

## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">Preprocessing</p>

<a id="2"></a>
<a href="#toc" class="btn btn-primary btn-sm" role="button" aria-pressed="true" 
style="color:blue; background-color:#dfa8e4" data-toggle="popover">Content</a>

## 2.1 Data Cleaning
Cleaning the dataset is critical before any analysis. Describe any issues encountered, such as:

- **Duplicates:** Remove duplicate records.
- **Inconsistent Formats:** Address inconsistent data formats (e.g., date formats, string casing).
- **Incorrect Data:** Handle obvious data errors (e.g., negative ages or future dates).

```python
# Example for Data Cleaning

# Remove Duplicates
df.drop_duplicates(inplace=True)

# Correct Inconsistent Formats (e.g., date format)
df['date_column'] = pd.to_datetime(df['date_column'])


### unnamed_0

In [25]:
df.drop(columns='unnamed_0', inplace=True)

### main_id

In [26]:
df['main_id'].value_counts(dropna=False)

main_id
751131ee    51
734264e5    47
d0778daf    39
6b00f3c8    36
aa6d52d8    36
            ..
50fef10a     1
db8a4949     1
2316a392     1
a8bc484a     1
8821da12     1
Name: count, Length: 28514, dtype: int64

### transaction id

In [27]:
df['transaction_id'].value_counts(dropna=False)

transaction_id
f56648eb    3
fe768cee    3
915a9d85    3
8eba900b    3
056168dd    3
           ..
4e0eb5ab    1
c9946c16    1
d1a35c5c    1
66f9b474    1
0ff8b41f    1
Name: count, Length: 48403, dtype: int64

### date

In [28]:
df['date'].value_counts(dropna=False)

date
2022-12-01 16:31:22.523    3
2022-10-04 20:53:56.160    3
2022-11-26 15:11:45.997    3
2022-10-18 17:56:53.440    3
2022-11-23 17:04:44.123    3
                          ..
2022-09-24 21:48:20.847    1
2022-11-18 19:49:01.973    1
2022-11-24 20:02:43.023    1
2022-11-06 13:07:01.423    1
2022-10-10 16:06:20.473    1
Name: count, Length: 48398, dtype: int64

### price

### code product

### amount

### item key

## 2.2 Missing Value Analysis
Evaluate the dataset for missing values:

- **Percentage of Missing Data:** Identify the percentage of missing data for each feature.
- **Handling Missing Data:** Explain the strategy used to handle missing values (e.g., removal, imputation using mean, median, mode, or more advanced methods).

## 2.3 Outlier Analysis
Identify and handle outliers in the data.Plot features using boxplots to visualize outliers.

## 2.4 Feature Engineering (if needed)
Transform or create new features to improve model performance

## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">Exploratory Data Analysis (EDA)</p>

<a id="3"></a>
<a href="#toc" class="btn btn-primary btn-sm" role="button" aria-pressed="true" 
style="color:blue; background-color:#dfa8e4" data-toggle="popover">Content</a>

## 3.1 Data Visualization
Visualize the data to identify trends, patterns, or anomalies. Suggested visualizations:

## 3.2 Correlation Analysis
Analyze correlations between numerical features

## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">4. Scaling, Categorical Variables, and Splitting</p>

<a id="4"></a>
<a href="#toc" class="btn btn-primary btn-sm" role="button" aria-pressed="true" 
style="color:blue; background-color:#dfa8e4" data-toggle="popover">Content</a>


## 4.1 Encoding Categorical Variables
Handle categorical features:

- **Label Encoding:** For ordinal variables.
- **One-Hot Encoding:** For nominal variables.

## 4.2 Splitting
Split the data into training and testing sets to avoid data leakage:

- **Train/Test Split:** Usually a 70/30 or 80/20 split.
- **Stratified Sampling:** If the dataset is imbalanced, ensure stratified sampling of the target variable.

## 4.3 Scaling
Normalize or standardize features to improve model performance, especially for distance-based algorithms (e.g., k-NN, SVM):

## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">5. Models</p>

<a id="5"></a>
<a href="#toc" class="btn btn-primary btn-sm" role="button" aria-pressed="true" 
style="color:blue; background-color:#dfa8e4" data-toggle="popover">Content</a>

## 5.1 Creating Models and Fine-Tuning
Build and evaluate baseline models using different machine learning algorithms. Improve model performance by tuning hyperparameters:

### 5.1.1 Logistic Regression (Ex Model 1)

## 5.2 Model Comparisons
Compare the performance of different models.

## 5.3 Feature Importance
Analyze and explain the most important features:

## 5.4 Final Model
Choose the best-performing model based on your evaluations and fine-tuning.

## 5.5 Create a Model with Fewer Features (if necessary)

## 5.6 Pickle the Model
Save the final model for future deployment:

## <p style="background-color:#fea162; font-family:newtimeroman; color:#FFF9ED; font-size:175%; text-align:center; border-radius:10px 10px;">6. Conclusion</p>

<a id="6"></a>
<a href="#toc" class="btn btn-primary btn-sm" role="button" aria-pressed="true" 
style="color:blue; background-color:#dfa8e4" data-toggle="popover">Content</a>

## Final Remarks
Summarize the key takeaways from the project. Highlight what was learned throughout the data science workflow and model deployment.

......